# 05 Fidelity: how closely does synthetic data match real training data?

**Purpose:** measure the statistical fidelity of every synthetic training set produced in `04_generators.ipynb`, using one method for all of them so results can be compared.

**Data rule:** each synthetic set is compared against **real `train.csv`** (with its genuine NaNs). **The test set is never loaded in this notebook.** Nothing is trained here.

**Experiments**

| Section | Strategy | Variant | Synthetic files compared |
|---|---|---|---|
| 1 | `random` | baseline, no target column | `{ctgan,tvae}_random_restored.csv` |
| 2 | `random` | target as generator column, unbalanced | `{ctgan,tvae}_random_target_restored.csv` |
| 3 | `from_column` | target as generator column, unbalanced | `{ctgan,tvae}_from_column_target_restored.csv` |
| 4 | all | side-by-side comparison | in memory |

**Metrics**
- **Column Shapes:** SDV quality report. The label is declared categorical where it exists.
- **Column Pair Trends:** mean `CorrelationSimilarity` over the six feature pairs, computed directly, because the SDV report returns `NaN` for this property in SDV 2.0.0 / SDMetrics 0.32.0 (evidence: `00_api_checks.ipynb`, section G). Each pair uses rows where both values are present. The label is excluded, because correlation is not defined for a categorical column.
- **Overall quality:** mean of Column Shapes and Pair Trends.
- **Missing-value similarity:** `MissingValueSimilarity` per feature, plus the raw difference in missing rate (percentage points).

**Comparability notes**
- Column Shapes averages **four** columns in the baseline and **five** in the target variants (the label scores near 1 and lifts the average mechanically). Compare baseline against target variants with care. The two target variants (sections 2 and 3) are like-for-like.
- Pair Trends covers the same four features everywhere.
- The manual pair-trends score follows the same idea as SDV's report but is not guaranteed identical to it, so the paper's values are for orientation only.

**Reproducibility check:** the saved numbers from earlier runs are listed in the setup cell, and each section prints whether the recomputed values match them.

**Outputs:** `reports/synthetic_quality_random_baseline.json`, `reports/synthetic_quality_random_target.json`, `reports/synthetic_quality_from_column_target.json`, `reports/fidelity_summary.json`.

In [1]:
import json
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path

import pandas as pd
import sdmetrics
import sdv
from sdmetrics.column_pairs import CorrelationSimilarity
from sdmetrics.single_column import MissingValueSimilarity
from sdv.evaluation import evaluate_quality
from sdv.metadata import Metadata

# ---------- Settings ----------
FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
GAP_COLS = ["nevents", "ndays_act", "nforum_posts"]   # features that contain missing values
TARGET = "Engagement_Level"
TABLE = "students"
GENERATORS = ("ctgan", "tvae")

# Values recorded in earlier runs (reproducibility check). from_column has none yet.
EXPECTED = {
    "random_baseline": {
        "ctgan": {"column_shapes": 0.8942, "pair_trends": 0.9394, "overall_quality": 0.9168},
        "tvae": {"column_shapes": 0.8643, "pair_trends": 0.9338, "overall_quality": 0.8990},
    },
    "random_target": {
        "ctgan": {"column_shapes": 0.9343, "pair_trends": 0.9624, "overall_quality": 0.9483},
        "tvae": {"column_shapes": 0.9141, "pair_trends": 0.9273, "overall_quality": 0.9207},
    },
}


# ---------- Helpers ----------
def find_project_root(start=None):
    """Walk upwards until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(f"Could not find configs/base.yaml above {start}")


def build_metadata(df: pd.DataFrame) -> Metadata:
    """Detect SDV metadata. The label, if present, is marked categorical."""
    metadata = Metadata.detect_from_dataframes({TABLE: df})
    if TARGET in df.columns:
        metadata.update_column(column_name=TARGET, table_name=TABLE, sdtype="categorical")
    return metadata


def column_shapes(real: pd.DataFrame, fake: pd.DataFrame, metadata) -> float:
    """Column Shapes score from the SDV quality report."""
    report = evaluate_quality({TABLE: real}, {TABLE: fake}, metadata, verbose=False)
    props = report.get_properties()
    return float(props.loc[props["Property"] == "Column Shapes", "Score"].iloc[0])


def pair_trends(real: pd.DataFrame, fake: pd.DataFrame) -> tuple[float, dict]:
    """Mean CorrelationSimilarity over all feature pairs (pairwise-complete rows)."""
    scores = {}
    for a, b in combinations(FEATURES, 2):
        pair = [a, b]
        scores[f"{a}|{b}"] = float(
            CorrelationSimilarity.compute(real[pair].dropna(), fake[pair].dropna())
        )
    return sum(scores.values()) / len(scores), scores


def evaluate_generator(real: pd.DataFrame, fake: pd.DataFrame, metadata) -> dict:
    """All fidelity metrics for one synthetic set against real train."""
    shapes = column_shapes(real, fake, metadata)
    trends, per_pair = pair_trends(real, fake)
    result = {
        "column_shapes": shapes,
        "pair_trends": trends,
        "overall_quality": (shapes + trends) / 2,
        "pair_trends_per_pair": per_pair,
        "missing_value_similarity": {
            c: float(MissingValueSimilarity.compute(real[c], fake[c])) for c in FEATURES
        },
        "missing_rate_difference_points": {
            c: round(float((fake[c].isna().mean() - real[c].isna().mean()) * 100), 2)
            for c in FEATURES
        },
    }
    if TARGET in fake.columns:
        share = fake[TARGET].value_counts(normalize=True).round(4).to_dict()
        result["class_share_synthetic"] = {str(k): v for k, v in share.items()}
    return result


def run_variant(key: str, strategy: str, variant: str, file_pattern: str, with_target: bool) -> dict:
    """Evaluate every generator for one experiment, print the scores and save the JSON."""
    cols = FEATURES + [TARGET] if with_target else FEATURES
    real = REAL_FULL[cols]
    metadata = build_metadata(real)

    results = {
        "sdv": sdv.__version__, "sdmetrics": sdmetrics.__version__,
        "strategy": strategy, "variant": variant,
        "compared_against": "real train (with NaNs)",
        "note": "pair trends computed manually over the 4 features: SDV report returned NaN",
        "generators": {},
    }

    for name in GENERATORS:
        path = GEN_DIR / file_pattern.format(name=name)
        if not path.exists():
            print(f"{name}: {path.name} not found, skipped")
            continue
        fake = pd.read_csv(path)[cols]
        res = evaluate_generator(real, fake, metadata)
        results["generators"][name] = res

        print(f"=== {name.upper()} | {strategy} | {variant} ===")
        print(f"Column Shapes : {res['column_shapes']:.4f}")
        print(f"Pair Trends   : {res['pair_trends']:.4f}")
        print(f"Overall       : {res['overall_quality']:.4f}")
        print("Per pair      :", {k: round(v, 3) for k, v in res["pair_trends_per_pair"].items()})
        print("Missing sim.  :", {k: round(v, 4) for k, v in res["missing_value_similarity"].items()})
        print("Rate diff (pts):", res["missing_rate_difference_points"])
        if "class_share_synthetic" in res:
            print("Class share   :", res["class_share_synthetic"])

        expected = EXPECTED.get(key, {}).get(name)
        if expected:
            ok = all(abs(res[m] - v) < 5e-4 for m, v in expected.items())
            print("Reproducibility check vs earlier run:", "match" if ok else "DIFFERS")
        print()

    out = REPORT_DIR / f"synthetic_quality_{key}.json"
    out.write_text(json.dumps(results, indent=2), encoding="utf-8")
    print("Saved:", out.name)
    return results


# ---------- Paths and shared inputs ----------
PROJECT_ROOT = find_project_root()
GEN_DIR = PROJECT_ROOT / "artifacts" / "generators"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

REAL_FULL = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")[FEATURES + [TARGET]]

stale = REPORT_DIR / "synthetic_quality_random.json"
if stale.exists():
    stale.unlink()
    print("Removed invalid file from the first evaluation:", stale.name)

all_results = {}
print("sdv:", sdv.__version__, "| sdmetrics:", sdmetrics.__version__)
print("Project root:", PROJECT_ROOT)
print("Real train:", REAL_FULL.shape, "| NaN % per feature:",
      (REAL_FULL[FEATURES].isna().mean() * 100).round(2).to_dict())

sdv: 2.0.0 | sdmetrics: 0.32.0
Project root: D:\student_engagement_twin
Real train: (21328, 5) | NaN % per feature: {'nevents': 17.08, 'ndays_act': 10.53, 'ncontent': 0.0, 'nforum_posts': 68.11}


## 1. Baseline: `random` strategy, no target column

**Goal:** fidelity of the spec-compliant baseline, where the generators never saw the label.

**Comparison:** real train features (four columns) against `ctgan_random_restored.csv` and `tvae_random_restored.csv`. The label does not exist in these files, so Column Shapes covers four columns and no class share is reported.

**Output:** `reports/synthetic_quality_random_baseline.json`.

**Expected (earlier run):** CTGAN overall 0.9168, TVAE 0.8990. Missing-value similarity about 1.0 (the NaNs were restored at the real rates).

In [2]:
all_results["random_baseline"] = run_variant(
    key="random_baseline", strategy="random", variant="no-target baseline",
    file_pattern="{name}_random_restored.csv", with_target=False,
)

=== CTGAN | random | no-target baseline ===
Column Shapes : 0.8942
Pair Trends   : 0.9394
Overall       : 0.9168
Per pair      : {'nevents|ndays_act': 0.961, 'nevents|ncontent': 0.949, 'nevents|nforum_posts': 0.884, 'ndays_act|ncontent': 0.957, 'ndays_act|nforum_posts': 0.979, 'ncontent|nforum_posts': 0.906}
Missing sim.  : {'nevents': 0.9996, 'ndays_act': 0.9995, 'ncontent': 1.0, 'nforum_posts': 0.9962}
Rate diff (pts): {'nevents': 0.04, 'ndays_act': -0.05, 'ncontent': 0.0, 'nforum_posts': 0.38}
Reproducibility check vs earlier run: match

=== TVAE | random | no-target baseline ===
Column Shapes : 0.8643
Pair Trends   : 0.9338
Overall       : 0.8990
Per pair      : {'nevents|ndays_act': 0.964, 'nevents|ncontent': 0.893, 'nevents|nforum_posts': 0.899, 'ndays_act|ncontent': 0.955, 'ndays_act|nforum_posts': 0.898, 'ncontent|nforum_posts': 0.993}
Missing sim.  : {'nevents': 0.9996, 'ndays_act': 0.9995, 'ncontent': 1.0, 'nforum_posts': 0.9962}
Rate diff (pts): {'nevents': 0.04, 'ndays_act'

## 2. `random` strategy, target as generator column (unbalanced)

**Goal:** fidelity of the paper's main setting, where the label is a categorical generator column and NaNs were restored at the real training rates.

**Comparison:** real train features and label against `{ctgan,tvae}_random_target_restored.csv`.

**Output:** `reports/synthetic_quality_random_target.json`.

**Expected (earlier run):** CTGAN overall 0.9483 and TVAE 0.9207. Class share close to the real 0.8246 / 0.1754 (CTGAN 0.824 / 0.176, TVAE 0.8435 / 0.1565).

In [3]:
all_results["random_target"] = run_variant(
    key="random_target", strategy="random", variant="target as generator column (unbalanced)",
    file_pattern="{name}_random_target_restored.csv", with_target=True,
)

=== CTGAN | random | target as generator column (unbalanced) ===
Column Shapes : 0.9343
Pair Trends   : 0.9624
Overall       : 0.9483
Per pair      : {'nevents|ndays_act': 0.993, 'nevents|ncontent': 0.978, 'nevents|nforum_posts': 0.918, 'ndays_act|ncontent': 0.984, 'ndays_act|nforum_posts': 0.988, 'ncontent|nforum_posts': 0.913}
Missing sim.  : {'nevents': 0.9996, 'ndays_act': 0.9995, 'ncontent': 1.0, 'nforum_posts': 0.9962}
Rate diff (pts): {'nevents': 0.04, 'ndays_act': -0.05, 'ncontent': 0.0, 'nforum_posts': 0.38}
Class share   : {'0': 0.824, '1': 0.176}
Reproducibility check vs earlier run: match

=== TVAE | random | target as generator column (unbalanced) ===
Column Shapes : 0.9141
Pair Trends   : 0.9273
Overall       : 0.9207
Per pair      : {'nevents|ndays_act': 0.97, 'nevents|ncontent': 0.888, 'nevents|nforum_posts': 0.83, 'ndays_act|ncontent': 0.998, 'ndays_act|nforum_posts': 0.968, 'ncontent|nforum_posts': 0.909}
Missing sim.  : {'nevents': 0.9996, 'ndays_act': 0.9995, 'ncont

## 3. `from_column` strategy, target as generator column (unbalanced)

**Goal:** fidelity of the second missingness strategy, where the generators modelled the missingness flags and the generated flags restored the NaNs.

**Comparison:** real train features and label against `{ctgan,tvae}_from_column_target_restored.csv`. This is like-for-like with section 2: same columns, same metrics, same real reference.

**What to look at:** the missing-value similarity and the `Rate diff (pts)` line. In `04_generators` the flag rates deviated from real by up to 10 points (CTGAN `nevents` +8.7, TVAE `ndays_act` -10.2), so missing-value similarity is expected to be lower than in section 2. Whether Column Shapes and Pair Trends change is read from the output, not predicted.

**Output:** `reports/synthetic_quality_from_column_target.json`. There is no earlier run to compare with.

In [4]:
all_results["from_column_target"] = run_variant(
    key="from_column_target", strategy="from_column",
    variant="target as generator column (unbalanced)",
    file_pattern="{name}_from_column_target_restored.csv", with_target=True,
)

=== CTGAN | from_column | target as generator column (unbalanced) ===
Column Shapes : 0.9433
Pair Trends   : 0.9587
Overall       : 0.9510
Per pair      : {'nevents|ndays_act': 0.99, 'nevents|ncontent': 0.939, 'nevents|nforum_posts': 0.95, 'ndays_act|ncontent': 0.976, 'ndays_act|nforum_posts': 0.997, 'ncontent|nforum_posts': 0.901}
Missing sim.  : {'nevents': 0.9129, 'ndays_act': 0.9611, 'ncontent': 1.0, 'nforum_posts': 0.9728}
Rate diff (pts): {'nevents': 8.71, 'ndays_act': -3.89, 'ncontent': 0.0, 'nforum_posts': 2.72}
Class share   : {'0': 0.8352, '1': 0.1648}

=== TVAE | from_column | target as generator column (unbalanced) ===
Column Shapes : 0.9297
Pair Trends   : 0.9567
Overall       : 0.9432
Per pair      : {'nevents|ndays_act': 0.936, 'nevents|ncontent': 0.896, 'nevents|nforum_posts': 0.96, 'ndays_act|ncontent': 0.996, 'ndays_act|nforum_posts': 0.998, 'ncontent|nforum_posts': 0.953}
Missing sim.  : {'nevents': 0.9801, 'ndays_act': 0.8984, 'ncontent': 1.0, 'nforum_posts': 0.9651

## Results: fidelity of `from_column` generators (target variant, unbalanced)

**Setup:** `{ctgan,tvae}_from_column_target_restored.csv` compared with real train (genuine NaNs). Same method as the `random` target variant. Saved in `reports/synthetic_quality_from_column_target.json`.

**Measured**

| Metric | CTGAN | TVAE |
|---|---|---|
| Column Shapes (5 columns incl. label) | 0.9433 | 0.9297 |
| Column Pair Trends (manual) | 0.9587 | 0.9567 |
| Overall quality | 0.9510 | 0.9432 |
| Missing-value similarity (nevents / ndays_act / ncontent / nforum_posts) | 0.9129 / 0.9611 / 1.0 / 0.9728 | 0.9801 / 0.8984 / 1.0 / 0.9651 |
| Missing-rate difference vs real, points | +8.71 / -3.89 / 0 / +2.72 | -1.99 / -10.16 / 0 / -3.49 |
| Class share (0 / 1) | 0.8352 / 0.1648 | 0.8296 / 0.1704 |

**Interpretation**
- Overall quality is higher than under `random` for both generators (CTGAN +0.003, TVAE +0.023). The CTGAN difference is small and its pair trends are slightly lower, so it is not interpreted. One seed.
- Missing-value similarity is lower than under `random` (mean over the three columns with gaps about 0.948 against 0.9985). The `random` figure is near-perfect by construction and validates the restoration only. The `from_column` figure reflects what the generators learned.
- The weakest points are the flag rates: CTGAN `nevents` (+8.7 points) and TVAE `ndays_act` (-10.2 points).
- Hypothesis (untested): the Column Shapes gain may come from `from_column` missingness depending on feature values, which brings the observed distributions closer to the real ones.

**Comparison with the paper (reference only):** from-column overall quality in the paper is CTGAN 91.94%, TVAE 92.24%. Ours are 95.10% and 94.32%. Direction agrees (from-column above random), but the pair-trends calculation is manual and the setups may differ, so these are not directly comparable.

**Limitations:** one seed per generator; manual pair trends are not guaranteed identical to SDV's.

## 4. Side-by-side comparison and summary

**Goal:** one table with every generator and strategy, and a saved summary for the report and the dashboard.

**Columns**
- `shapes`, `pairs`, `overall`: as defined above.
- `missing_sim_mean`: mean missing-value similarity over the three features that have gaps (`ncontent` has none and always scores 1.0, so it is left out to avoid inflating the mean).

**Reading the table**
- The rows `random_target` and `from_column_target` are like-for-like.
- `random_baseline` has four-column Column Shapes. Do not compare its `shapes` and `overall` directly with the target rows.
- One seed per generator, so small differences are not interpreted as winners.

**Paper values (reference only, not targets):** random strategy overall quality CTGAN 83.55%, TVAE 89.92%. From-column CTGAN 91.94%, TVAE 92.24%. The paper's random-strategy missing-value similarity is 0.9983 / 0.9983 / 0.9954. Our pair-trends calculation is manual, and our setup may differ from the paper's, so these are not directly comparable.

**Output:** `reports/fidelity_summary.json`.

In [5]:
rows = {}
for key, res in all_results.items():
    for name, g in res["generators"].items():
        sim = g["missing_value_similarity"]
        rows[f"{name} | {key}"] = {
            "shapes": g["column_shapes"],
            "pairs": g["pair_trends"],
            "overall": g["overall_quality"],
            "missing_sim_mean": sum(sim[c] for c in GAP_COLS) / len(GAP_COLS),
        }

summary = pd.DataFrame(rows).T.round(4)
print(summary.to_string())

payload = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "versions": {"sdv": sdv.__version__, "sdmetrics": sdmetrics.__version__, "pandas": pd.__version__},
    "compared_against": "real train (with NaNs); test set not used",
    "summary": summary.to_dict(orient="index"),
    "experiments": list(all_results),
}
(REPORT_DIR / "fidelity_summary.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("\nSaved:", REPORT_DIR / "fidelity_summary.json")

                            shapes   pairs  overall  missing_sim_mean
ctgan | random_baseline     0.8942  0.9394   0.9168            0.9985
tvae | random_baseline      0.8643  0.9338   0.8990            0.9985
ctgan | random_target       0.9343  0.9624   0.9483            0.9985
tvae | random_target        0.9141  0.9273   0.9207            0.9985
ctgan | from_column_target  0.9433  0.9587   0.9510            0.9489
tvae | from_column_target   0.9297  0.9567   0.9432            0.9479

Saved: D:\student_engagement_twin\reports\fidelity_summary.json


## Results: fidelity summary (all experiments)

| Generator \| experiment | Column Shapes | Pair Trends | Overall | Mean missing-value similarity |
|---|---|---|---|---|
| CTGAN \| random baseline (4 columns) | 0.8942 | 0.9394 | 0.9168 | 0.9985 |
| TVAE \| random baseline (4 columns) | 0.8643 | 0.9338 | 0.8990 | 0.9985 |
| CTGAN \| random, target | 0.9343 | 0.9624 | 0.9483 | 0.9985 |
| TVAE \| random, target | 0.9141 | 0.9273 | 0.9207 | 0.9985 |
| CTGAN \| from_column, target | 0.9433 | 0.9587 | 0.9510 | 0.9489 |
| TVAE \| from_column, target | 0.9297 | 0.9567 | 0.9432 | 0.9479 |

**Reading the table**
- The target rows (`random` and `from_column`) are like-for-like. The baseline rows average Column Shapes over four columns, so their `shapes` and `overall` are not directly comparable with the target rows.
- CTGAN scores higher than TVAE in every row of overall quality. The gap shrinks under `from_column` (0.008), and it is one seed.
- The missing-value similarity of 0.9985 in the `random` rows reflects the restoration procedure, not generator quality.

**Limitations:** one seed per generator; the manual pair-trends score is not guaranteed identical to SDV's.

**Outputs:** `reports/synthetic_quality_{random_baseline,random_target,from_column_target}.json`, `reports/fidelity_summary.json`.